[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/23_cross_attention.ipynb)

# 🟠 Medium: Multi-Head Cross-Attention

Implement **multi-head cross-attention** (encoder-decoder attention).

### Signature
```python
class MultiHeadCrossAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, x_q: Tensor, x_kv: Tensor) -> Tensor:
        # x_q: (B, S_q, D) — decoder queries
        # x_kv: (B, S_kv, D) — encoder keys/values
```

### Key Differences from Self-Attention
- Q comes from the decoder, K and V come from the encoder
- No causal mask (all encoder positions visible)

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 1.9 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn
import math

In [5]:
# ✏️ YOUR IMPLEMENTATION HERE

class MultiHeadCrossAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.W_q = nn.Linear(d_model, d_model) # x (B, S_q, D), x@W_q -> (B, S_q, D) then split by head
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.head_dim = d_model // num_heads
        self.num_heads = num_heads
        self.d_model = d_model
        pass  # W_q, W_k, W_v, W_o

    def forward(self, x_q, x_kv):
        B, S_q = x_q.shape[0], x_q.shape[1]
        S_kv = x_kv.shape[1]
        Q = self.W_q(x_q).reshape(B, S_q, self.num_heads, -1).permute(0, 2, 1, 3) # (B, h, S_q, head_dim)
        K = self.W_k(x_kv).reshape(B, S_kv, self.num_heads, -1).permute(0, 2, 1, 3)
        V = self.W_v(x_kv).reshape(B, S_kv, self.num_heads, -1).permute(0, 2, 1, 3) # (B, h, S_kv, head_dim)

        # Q@K^T / sqrt(dk)
        attn_logits = torch.einsum("bhqd,bhkd->bhqk", Q, K) / math.sqrt(self.head_dim) # # (B, h, S_q, S_kv)
        attn_score = nn.functional.softmax(attn_logits, dim=-1)
        attn_out = torch.einsum("bhqk,bhkd->bqhd", attn_score, V).reshape(B, S_q, self.d_model) ## (B, h, S_q, head_dim) -> (B, sq, h, head_dim) -> (B, sq, D)
        return self.W_o(attn_out)

        pass  # Q from x_q, K/V from x_kv, no causal mask

In [6]:
# 🧪 Debug
attn = MultiHeadCrossAttention(64, 4)
x_q = torch.randn(2, 6, 64)
x_kv = torch.randn(2, 10, 64)
print('Output:', attn(x_q, x_kv).shape)

Output: torch.Size([2, 6, 64])


In [7]:
# ✅ SUBMIT
from torch_judge import check
check('cross_attention')


🧪 Testing: Multi-Head Cross-Attention (Medium)
──────────────────────────────────────────────────
  ✅ [1/4] Output shape (3.1ms)
  ✅ [2/4] Q and KV different lengths (1.3ms)
  ✅ [3/4] No causal mask — all KV affects all Q (40.1ms)
  ✅ [4/4] Gradient flow (27.9ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (72.4ms total)
  Progress saved. Run status() to see your dashboard.

